# 1차 실행 -  논문 목록 다운로드

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv

# 1. 명시적으로 .env 파일 로드 (파일명 확인 필수!)
if os.path.exists('.env'):
    load_dotenv('.env', override=True) # 기존 환경변수보다 .env 파일 내용을 우선함
else:
    raise FileNotFoundError(".env 파일이 없습니다. .env.sample을 복사해서 .env를 만들어주세요.")

API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 키 값이 샘플 값인지 확인하는 안전장치
if not API_KEY or "여기에" in API_KEY:
    raise ValueError("API_KEY가 설정되지 않았거나 샘플 값입니다. .env 파일을 확인하세요.")

def fetch_articles_by_journal(api_key, journal_name):
    articles = []
    page = 1
    display_count = 100
    
    while True:
        params = {
            "apiCode": "articleSearch",
            "key": api_key,
            "journal": journal_name,
            "displayCount": display_count,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
            root = ET.fromstring(response.content)

            # [핵심] 결과 메시지 확인 (키 오류 등 확인용)
            result_msg_node = root.find(".//{*}resultMsg")
            if result_msg_node is not None and "정상" not in result_msg_node.text:
                # '등록되지 않은 key입니다' 등이 여기서 걸러짐
                print(f"\n[API 메시지 - {journal_name}]: {result_msg_node.text}")
                break

            # 결과 개수 확인
            total_node = root.find(".//{*}total")
            if total_node is None or int(total_node.text) == 0:
                break
                
            total_count = int(total_node.text)
            
            # 레코드 추출 시작
            records = root.findall(".//{*}record")
            for record in records:
                try:
                    article_info = record.find("{*}articleInfo")
                    journal_info = record.find("{*}journalInfo")
                    if article_info is None: continue

                    # 데이터 파싱 시 None 방지 (findtext 활용)
                    articles.append({
                        "학술지명": journal_name,
                        "논문ID": article_info.get("article-id", ""),
                        "제목": article_info.findtext(".//{*}article-title", default="제목 없음").strip(),
                        "저자": ", ".join([a.text for a in article_info.findall(".//{*}author") if a.text]),
                        "발행연도": journal_info.findtext("{*}pub-year", default="") if journal_info is not None else "",
                        "KCI_URL": article_info.findtext("{*}url", default="")
                    })
                except Exception:
                    continue # 개별 레코드 오류는 건너뜀
            
            if page * display_count >= total_count:
                break
            page += 1
            time.sleep(0.2) # 적절한 딜레이

        except Exception as e:
            print(f"\n[오류] {journal_name} 수집 중단: {e}")
            break
            
    return articles
# ==========================================
# 실행부
# ==========================================
if __name__ == "__main__":
    INPUT_FILE = '법학 기관 목록.csv'
    OUTPUT_FILE = '전체_법학_논문목록.csv'

    try:
        # 1. 데이터 로드
        df_org = pd.read_csv(INPUT_FILE, encoding='utf-8')
        if '학술지한글명' not in df_org.columns:
            raise ValueError("CSV 파일에 '학술지한글명' 칼럼이 없습니다.")
            
        target_journals = df_org['학술지한글명'].dropna().unique().tolist()
        print(f"총 {len(target_journals)}개의 학술지를 수집합니다.")

        # 2. 수집 진행
        all_results = []
        for jnl in tqdm(target_journals, desc="수집 진행률"):
            results = fetch_articles_by_journal(API_KEY, jnl)
            all_results.extend(results)
            
            # 중간 저장
            if len(all_results) % 500 == 0:
                pd.DataFrame(all_results).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

        # 3. 최종 저장
        if all_results:
            df_final = pd.DataFrame(all_results)
            df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n✅ 수집 완료! 총 {len(df_final)}건 저장됨.")
        else:
            print("\n❌ 수집된 데이터가 없습니다. API 키와 학술지명을 다시 확인하세요.")

    except Exception as e:
        print(f"프로그램 실행 중 치명적 오류: {e}")

### 디버그용 셀(실행 불필요)

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 테스트할 학술지명 하나 지정
test_journal = "법학연구" 

params = {
    "apiCode": "articleSearch",
    "key": API_KEY,
    "journal": test_journal,
    "displayCount": 1,
    "page": 1
}

try:
    response = requests.get(BASE_URL, params=params)
    print("="*50)
    print(f"HTTP 상태 코드: {response.status_code}")
    print("="*50)
    # 응답 내용 전체 출력
    print(response.text)
    print("="*50)
except Exception as e:
    print(f"요청 중 오류 발생: {e}")

# 중복제거

In [ ]:
import pandas as pd

INPUT_FILE = '전체_법학_논문목록.csv'
CLEANED_FILE = '전체_법학_논문목록_정제본.csv'

try:
    # 1. 데이터 로드
    df = pd.read_csv(INPUT_FILE)
    total_rows = len(df)
    
    # 2. '논문ID' 기준 중복 체크
    duplicate_mask = df.duplicated(subset=['논문ID'], keep=False)
    df_duplicates = df[duplicate_mask].sort_values(by='논문ID')
    
    # 3. 통계 계산
    unique_count = df['논문ID'].nunique()
    duplicate_count = total_rows - unique_count
    # 중복률 계산 (필요시)
    # $Duplication Rate = \frac{N_{duplicates}}{N_{total}} \times 100$
    
    print("="*50)
    print(f"📊 데이터 중복 체크 결과")
    print("-"*50)
    print(f" 전체 행(Row) 수: {total_rows}건")
    print(f" 고유 논문(Unique ID) 수: {unique_count}건")
    print(f" 중복된 데이터 수: {duplicate_count}건")
    print(f" 중복률: {(duplicate_count / total_rows * 100):.2f}%")
    print("="*50)

    # 4. 중복된 논문 샘플 출력 (중복이 있을 경우)
    if duplicate_count > 0:
        print("\n⚠️ 중복된 논문 샘플 (상위 5건):")
        print(df_duplicates[['논문ID', '제목', '학술지명']].head(10))
        
        # 5. 중복 제거 및 저장 선택
        # keep='first'를 써서 중복 중 첫 번째 행만 남김
        df_cleaned = df.drop_duplicates(subset=['논문ID'], keep='first')
        df_cleaned.to_csv(CLEANED_FILE, index=False, encoding='utf-8-sig')
        print(f"\n✅ 중복이 제거된 파일이 '{CLEANED_FILE}'로 저장되었습니다.")
    else:
        print("\n✅ 중복된 논문이 없습니다. 깨끗한 데이터입니다!")

except Exception as e:
    print(f"오류 발생: {e}")

# 세부 참고문헌 정보 다운로드

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 에러 발생식 재시도 전략
retry_strategy = Retry(
    total=5, # 최대 5번 재시도
    backoff_factor=2, # 재시도 간격 지수적 증가 (2초, 4초, 8초...)
    status_forcelist=[429, 500, 502, 503, 504], # 해당 상태 코드 시 재시도
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session = requests.Session()
session.mount("https://", adapter)

INPUT_FILE = '전체_법학_논문목록.csv'
OUTPUT_FILE = '법학_인용_네트워크_데이터.csv'

# 1. 기존에 수집된 데이터가 있다면 불러오기 (이어하기용)
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_existing = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(df_existing['source_id'].unique())
        print(f"이미 처리된 논문 {len(processed_ids)}건을 건너뜁니다.")
    except:
        pass

# 2. 대상 목록 로드 및 필터링
df_articles = pd.read_csv(INPUT_FILE)
article_ids = [aid for aid in df_articles['논문ID'].dropna().unique() if aid not in processed_ids]

def fetch_references_safe(article_id):
    references = []
    params = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    
    try:
        # timeout을 60초로 넉넉히 잡고 세션 사용
        response = session.get(BASE_URL, params=params, timeout=60)
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            ref_records = root.findall(".//{*}cited-literature/{*}record")
            for ref in ref_records:
                references.append({
                    "source_id": article_id,
                    "ref_title": ref.findtext("{*}title", default="").strip(),
                    "ref_author": ref.findtext("{*}author", default="").strip(),
                    "ref_year": ref.findtext("{*}pub-year", default="").strip(),
                    "ref_journal": ref.findtext("{*}journal", default="").strip()
                })
    except Exception as e:
        print(f"\n[건너뜀] {article_id} 오류: {e}")
    return references

# 3. 실행
all_results = []
for i, aid in enumerate(tqdm(article_ids, desc="인용 수집 중")):
    refs = fetch_references_safe(aid)
    if refs:
        all_results.extend(refs)
    
    # 50건마다 파일에 누적 저장
    if (i + 1) % 50 == 0:
        header = not os.path.exists(OUTPUT_FILE)
        pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')
        all_results = [] # 메모리 비우기
    
    time.sleep(2.0) # 지연 시간을 1초 이상으로 넉넉히 조절

# 남은 데이터 저장
if all_results:
    header = not os.path.exists(OUTPUT_FILE)
    pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')

인용 수집 중:   4%|▍         | 3938/88947 [2:21:21<49:56:30,  2.11s/it]